
## == output ===
## passes.csvz
## data.csv
## outcome.csv # timestamp grade outcome or what pass it is 


In [ ]:
#%pip install torch
#%pip install torch_geometric
print('hi')



#         Data Calculation & Data

Proccessing from CV to CSV

# Distance tp Nearest Player - distance_to_player(dataframe)

This is use to give a GAT more data and a better understand of where the players are on the pitch in relation to the ball


# Distance to Ball - distance_to_ball(dataframe)


# Loading Data

Here we need to now assingn the data to the Nodes, Edges and the Node Features and the Edge Features

# Need to review the and need to review how we write this path as we are loading the data and building the graph

# Could do build data then build graph and then Load Data in to DataLoader


Nodes - Players:

Node Features - Distance to ball:

Edges:

Edges Features:

Edge Index:

Edge Attr:



In [ ]:
# Refrance 1 Framework -  (GitHub, n.d.) - https://github.com/pyg-team/pytorch_geometric/blob/master/examples/gat.py

#https://www.youtube.com/watch?v=Gv0RT5N_mhg

import pandas as pd
import numpy
import time
import torch
import torch.nn as nn
import torch.nn.functional as F

import os
import math

from torch_geometric.nn import GATv2Conv, global_mean_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

device = torch.device('cpu')

## add tO export columns

path_1 = '/home/c3646202/Desktop/FootballPassingAnalysisProject2/output/data.csv' # Cheslea Vs Arsenal 10 Minutes
#path_2 = '' # Demo1 - 3o seconds

passes_csv_path = '/home/c3646202/Desktop/FootballPassingAnalysisProject2/pass_data_processing/passes.csv' 

def csv_import(path):
    dataframe = pd.read_csv(path)
    #print(dataframe.head())
    return dataframe

#------------------------------- 
#          Load Data In
#-------------------------------

passes_dataframe = pd.read_csv(passes_csv_path) # read data in
data_dataframe = pd.read_csv(path_1) 
y_grade = passes_dataframe[['pass_grade']]
pass_graph = []

#------------------------------- 
#          Build Graph 
#-------------------------------
# Graph Level Prediction - 1 graph per pass

## Helped build the graph - https://www.datacamp.com/tutorial/comprehensive-introduction-graph-neural-networks-gnns-tutorial?dc_referrer=https%3A%2F%2Fwww.google.com%2F


for index, rows in passes_dataframe.iterrows(): # creating the graph by going through each frame
    start_frame = rows['start_frame']
    end_frame = rows['end_frame']
    row = []
    column = []
# https://www.geeksforgeeks.org/python/how-to-fix-valueerror-the-truth-value-of-a-series-is-ambiguous-in-pandas/
    
    frame = data_dataframe[(data_dataframe['frame'] >= start_frame) & (data_dataframe['frame'] <= end_frame)]  ## validating if start and end frame are not before or after each other

    if frame.empty:
        continue
    ##frame =
    node_features = frame[['player_x', 'player_y', 'team', 'has_ball', 'ball_x', 'ball_y']]  ## Node Deature for that create the node
#edge_features = data_dataframe[['player_x', 'player_y']] ## add vy and vx as well 
    edge_features = frame[['distance_to_ball', 'has_ball', 'team']]
    team = frame['team'].values
    frame_length = len(frame)
    for i in range(frame_length):   ### this is gonna be spenny,  sliding window # get refrance
        for j in range(frame_length):
            if i != j: ## connects all player - for just teammates- if i != j and team[i] == team[j]:  ## this is only ball holder to temmates, need teammate, might need to change if I want cross team interaction can test
                row.append(i)
                column.append(j) 
    #   Data Formatting for Graph Attention Network 
    node_features = node_features.values.astype('float32') ## need to convert float32 to pytorch
    edge_features = edge_features.values[column].astype('float32') 
    edge_index = torch.tensor([row, column], dtype=torch.long)
    x = torch.tensor(node_features) #, dtype=torch.float
    #y_grade = torch.tensor(y_grade.to_numpy(), dtype=torch.long)# # dtype=torch.long # .squeeze() https://stackoverflow.com/questions/70173120/numpy-get-1d-array-from-a-2d-array-with-1-row
    if pd.isna(rows['pass_grade']): ## https://www.w3schools.com/python/pandas/ref_df_isna.asp
        continue
    percentage_to_grade = {0.1: 0, 0.3: 1, 0.5: 2, 0.7: 3, 0.9: 4} ## model only takes in whole numbers
    grade_value = percentage_to_grade[rows['pass_grade']]
    y_grade = torch.tensor([grade_value] , dtype=torch.long)# # dtype=torch.long # from 0-4 to 1-5# fhttps://stackoverflow.com/questions/70173120/numpy-get-1d-array-from-a-2d-array-with-1-row
    edge_attr = torch.tensor(edge_features)
    
    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y_grade=y_grade,) #edge_attr=edge_attr
    pass_graph.append(data)


    
        # size in each edge features that you pass into the model, -varible you pass into edge_attr 
    #need to break this down edge_features is just an array of feature ve for each edge, and column represents the indices of the edges, then your edge_attr is correct
    #https://www.kaggle.com/code/rafsunsheikh/convert-any-tabular-data-to-graph-for-gnn
    
    ### Creating One Object Per Pass to Calculate One Pass per Graph - https://pytorch-geometric.readthedocs.io/en/2.5.2/get_started/introduction.html#data-transforms





# Graph Attention Network - GATv2

Paramaters:

Nodes - node features
Nodes features – Player features and ball features, such as 
Edge features – Distance to the ball, distance to nearest player, ball_speed
Edge_index – Row and columns of the 
Edge_attr - redg

Layers:

Layer 1:
Player's Atrtributes
Layer 2:
Player to Player Connections
Layer 3:
Team to Team Connections

Norms:

Foaward:

Training:

Exporting the Data:





In [2]:


loader = DataLoader(pass_graph, batch_size= 4, shuffle= False) ## get ref # Loading in the data from the build graph
##----------------------------------------------------------

#----------------------------------
#             Config
#----------------------------------

in_channels = 6# OR IS THIS 7??  # check this +++ can we get this to 16, will it be faster??
hidden_channels  = 64   # hiddel and out_channels
Dropout= 0.2 # start at 0.2 then go up if needed
per_head = hidden_channels // 2
num_grade_classes = 5 # blunder, mistake, textbook, good, brilliant
compress_channel = 32
export_data_model_path = '/home/c3646202/Desktop/FootballPassingAnalysisProject2/models/model.pt'

#----------------------------------
#             Model
#----------------------------------

# Refrances:
# 1 https://github.com/pyg-team/pytorch_geometric/blob/master/examples/gat.py 
# supporting # https://pytorch-geometric.readthedocs.io/en/latest/get_started/introduction.html#exercises - check

class PassingGAT(nn.Module):
    def __init__(self):  ## define here self, in_channels, hidden_channels, out_channels, heads):
        super().__init__()
        # follow this format - in_channels, hidden_channels, heads, dropout=0.6
        self.conv1 = GATv2Conv(in_channels, per_head , heads=2, edge_dim=3, concat=True, dropout=Dropout)
        self.norm1 = nn.LayerNorm(hidden_channels)

        self.conv2 = GATv2Conv(hidden_channels, per_head, heads=2, edge_dim=3, concat=True, dropout=Dropout)
        self.norm2 = nn.LayerNorm(hidden_channels)

        self.conv3 = GATv2Conv(hidden_channels, hidden_channels, heads=1, edge_dim=3, concat=False, dropout=Dropout)

        # Residual projection
        self.res_proj = nn.Linear(in_channels, hidden_channels)
       # self.encoder = nn.Linear(hidden_channels, compress_channel) # https://www.geeksforgeeks.org/deep-learning/implementing-an-autoencoder-in-pytorch/ - in out or out in ?? - https://python.pages.doc.ic.ac.uk/lessons/pytorch/08-autoencoder/04-refactoring-ae.html - 64 64
       # self.decoder = nn.Linear(compress_channel , in_channels) # https://python.pages.doc.ic.ac.uk/lessons/pytorch/08-autoencoder/02-auto-encoder.html 64 11?? not really getting compressed does make sense maybe half??

        # Heads - no longer needed ????
        self.grade_head   = nn.Linear(hidden_channels, num_grade_classes)
 
    def forward(self, x, edge_index, batch, edge_attr):

        # Residual connection
        res = self.res_proj(x)

        # Layer 1
        x = self.conv1(x, edge_index, edge_attr)
        x = F.elu(x)
        x = self.norm1(x)
        x = F.dropout(x, p=Dropout, training=self.training)
        # Layer 2
        x = self.conv2(x, edge_index, edge_attr)
        x = F.elu(x)
        x = self.norm2(x)
        x = F.dropout(x, p=Dropout, training=self.training)
        # Layer 3
        x = self.conv3(x, edge_index, edge_attr)

        # Add residual impor
        x = x + res ## need to change this as this could break if dimention of modle change
        #compress = F.relu((self.encoder(x)))
        #x_uncompressed = self.decoder(compress) ### check this 



        #Reconsruction of the data
        x = global_mean_pool(x, batch) # from 40, 64 to  1, 64 one grade per pass not graph
       # predict
        grade = self.grade_head(x)

        return grade # outcome  # could use cross entropy loss but use this and nll_
device = torch.device('cpu')   
model = PassingGAT().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)

#------------------------------- 
#           Train
#-------------------------------

def export_data(model):
    torch.save(model.state_dict(), export_data_model_path) ### Refrance https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html
    #grades.to_csv('ML_Results.csv', index=False)

##  Train
def train(data, model, device):
    model.train()
    optimizer.zero_grad()
    grade = model(data.x, data.edge_index, data.batch, data.edge_attr) 
    # https://stackoverflow.com/questions/69965519/cross-entropy-loss-argument-target-position-2-must-be-tensor-not-numpy-n
    print(data.x.shape)
    ## = F.mse_loss(x_uncompressed, data.x) # input, target https://python.pages.doc.ic.ac.uk/lessons/pytorch/08-autoencoder/02-auto-encoder.html, from this found this https://docs.pytorch.org/docs/stable/generated/torch.nn.MSELoss.html#torch.nn.MSELoss
    grade_loss = F.cross_entropy(grade, data.y_grade.view(-1)) ## ref - https://discuss.pytorch.org/t/valueerror-expected-input-batch-size-1-to-match-target-batch-size-4-how-can-i-fix-this/202084
    loss = grade_loss
    loss.backward()
    optimizer.step()
    return float(loss.detach()) # need to change back to return total_loss.detech() or item()


#------------------------------- 
#            Test
#-------------------------------

grades_data = []
@torch.no_grad()  ## do we need this
def test(loader, model, device):
    device = torch.device('cpu')
    model.eval()
    accs = []   
    for data in loader:
        data = data.to(device)
        grade = model(data.x, data.edge_index, data.batch, data.edge_attr) #https://halil7hatun.medium.com/graph-neural-networks-gnns-1f463df4bb77
        grade_predictions = grade.argmax(dim=1)
        correct_predictions = int((grade_predictions == data.y_grade).sum()) ##
        total = data.y_grade.shape[0]
        accs.append(correct_predictions/total)
        
        # Convert From Numpy Autoencoder to Numpy for KMeans
        #compress = compress.cpu().detach().numpy()# https://stackoverflow.com/questions/49768306/pytorch-tensor-to-numpy-array
      #  kmeans = KMeans(n_clusters=5, init='k-means++', random_state=15, n_init=1) ## using 5 clusters as 5 different classication could use the elbow algo to decide my clusters as well check # https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html, 5 cluster becuase we have 5 different catergorise
       # grades = kmeans.fit_predict(compress) + 1 ## check varible name
       # print(f'grades: {grade_predictions}')
        # Do I need to swithc back from Numpy
    return accs


#------------------------------- 
#          Training Loop
#-------------------------------
 
def training(loader, model, device, epoch=100):
    times = []
    for epoch in range(1, epoch + 1): ##change this back to 200 ## if adding para backing change to 1, args.epochs + 1 - args start at 0, deault = 200
        start = time.time()
        for data in loader:
            data = data.to(device)
            loss = train(data, model, device)
            print(f'loss: {loss}')
        accs = test(loader, model, device)
        print(f'grade : {accs}')
        times.append(time.time() - start)
    return times, accs, loss


def display_ouput(times, accs, epoch, loss, model):
    print(f"Median time per epoch: {torch.tensor(times).median():.4f}s")
    print(f'Grade Accuracy: {accs} Epoch: {epoch}, Loss: {loss}')
    print(f'\n===================================================================\n \t\tExporting \n===================================================================\n') 
  #  print(f'\tPassing Grade Results:\n\n{grades}\n\n')
    print(f'\tModel:\n\n{model}\n \n===================================================================\n') ## does it need to be a pickle can't it just be a txt ??? 
    export_data(model) ### why does it need to be a pickle can't it just be a txt?

def main():
    model.to(device)  ## sending the model to th GPU or CPU
    times, accs, loss = training(loader, model, device, epoch=100) 
    display_ouput(times, accs, 100, loss, model)

main()

## check what we are actuall exporting is accurate
## do webiste automation .sh
## need to do model when once train model and importing that and exporting the result
    
    
## Problem--------------------------------

## to loader check why needed numpy for loader 



##might of forgot to add refrance for them in here check I am espicall when it goes from pytorch tensor to numpy



## Next steps ---------------------------------------------------------

## So done pre tuning 
## Could do fine-tuning
## Could do instruct-tuning --- prompt tuning


## need to remove black output box 


torch.Size([2127, 6])
loss: 906.40771484375
torch.Size([8210, 6])
loss: 433.104736328125
torch.Size([10801, 6])
loss: 572.796630859375
torch.Size([7175, 6])
loss: 469.0387878417969
grade : [0.0, 0.25, 0.0, 0.0]
torch.Size([2127, 6])
loss: 702.9129638671875
torch.Size([8210, 6])
loss: 313.359375
torch.Size([10801, 6])
loss: 392.460693359375
torch.Size([7175, 6])
loss: 270.3263854980469
grade : [0.0, 0.25, 0.0, 0.0]
torch.Size([2127, 6])
loss: 513.9014892578125
torch.Size([8210, 6])
loss: 198.354248046875
torch.Size([10801, 6])
loss: 216.00982666015625
torch.Size([7175, 6])
loss: 74.38728332519531
grade : [0.0, 0.25, 0.25, 0.0]
torch.Size([2127, 6])
loss: 333.1299133300781
torch.Size([8210, 6])
loss: 91.77037048339844
torch.Size([10801, 6])
loss: 80.02867126464844
torch.Size([7175, 6])
loss: 0.0
grade : [0.25, 0.25, 0.75, 1.0]
torch.Size([2127, 6])
loss: 302.9114074707031
torch.Size([8210, 6])
loss: 105.16419982910156
torch.Size([10801, 6])
loss: 67.78107452392578
torch.Size([7175, 6])
l

In [4]:
# Proccessing CSV to Metrics / ## ML Model Config


%whos

Variable                 Type          Data/Info
------------------------------------------------
Data                     ABCMeta       <class 'torch_geometric.data.data.Data'>
DataLoader               type          <class 'torch_geometric.l<...>r.dataloader.DataLoader'>
Dropout                  float         0.2
F                        module        <module 'torch.nn.functio<...>/torch/nn/functional.py'>
GATv2Conv                type          <class 'torch_geometric.n<...>nv.gatv2_conv.GATv2Conv'>
PassingGAT               type          <class '__main__.PassingGAT'>
column                   list          n=263682
compress_channel         int           32
csv_import               function      <function csv_import at 0x7f17b5f05760>
data                     Data          Data(x=[7175, 6], edge_in<...>1473450, 3], y_grade=[1])
data_dataframe           DataFrame     Shape: (213947, 10)
device                   device        cpu
display_ouput            function      <function display_ou

In [ ]:
# Run 2
##https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html
load_model = torch.load('model.pt')
model.load_state_dict(load_model['model_state_dict'])


#model = PassingGAT().to(device)
#model.load_state_dict(torch.load('model.pt', weights_only=True))
#model.eval()

In [ ]:
# New CODE 

## Spair Code Might add in later 


### So you can run on the command lin
parser = argparse.ArgumentParser()
parser.add_argument('--dataset', type=str, default='Cora')
parser.add_argument('--hidden_channels', type=int, default=8)
parser.add_argument('--heads', type=int, default=8)
parser.add_argument('--lr', type=float, default=0.005)
parser.add_argument('--epochs', type=int, default=200)
parser.add_argument('--wandb', action='store_true', help='Track experiment')
args = parser.parse_args()


#### goes to a dashboard which we need
init_wandb(name=f'GAT-{args.dataset}', heads=args.heads, epochs=args.epochs,
           hidden_channels=args.hidden_channels, lr=args.lr, device=device)

path = osp.join(osp.dirname(osp.realpath(__file__)), '..', 'data', 'Planetoid')
dataset = Planetoid(path, args.dataset, transform=T.NormalizeFeatures())
data = dataset[0].to(device)


In [ ]:
# W.02: The Poisson Distribution

\pagetab{W.02}

# Assumptions

Assuming that all loom are the same to a certain extent, yet they could always fatigue at different rates


In [ ]:
## Comments ##############
    ## Is the attr_ correct?? - actually check all varibles
    ## drop out could be redundant


# for the for loop need to test cross team interactions as well

 ## what do we use this for is it data below at model eval
##currently not working still :)))

# moght need to scale or normalisation

#  x = x + res ## need to change this as this could break if dimention of modle change

## found good repo for project put this in documentation how you were having alot of problem and then releasise this was unsup, instead of semi

## but from this you really learned ml 
## and though to check there github and you are now going to use there model.

In [ ]:

## Loooking at problem right now think it is the patch but dont think it is


In [ ]:
## Citations that could be good for my project


# Direct:
# - https://yashvaantlakham73.medium.com/learn-autoencoders-and-graph-neural-networks-from-scratch-03ab884b2434 - use there image in report

# Indirectly / somewhat used  
# - https://github.com/pyg-team/pytorch_geometric/blob/master/examples/autoencoder.py - Fey, M and Lenssen

# Was going to use PyTorch structure but did not work well for - https://discuss.pytorch.org/t/graph-reconstruction-using-autoencoder/221320 and kinda has the same use case as me
# his project ^ vs my project emmbedded graph of passes embedded a graph of passes and use these to classificy task in this case passing scores

# node - players
# edgesa - passes between players 
# node atrritbutes - player stats and position
# edge attribute - dustance, outcome

# so would go GATv2Conv -> graph level embedding -> enodcode = reconstruction for adjency matrix + Node attributes#


# use the gat.py for the base have specfic ref
# loader
# medium - https://yashvaantlakham73.medium.com/learn-autoencoders-and-graph-neural-networks-from-scratch-03ab884b2434
# autoencoder - https://www.geeksforgeeks.org/deep-learning/implementing-an-autoencoder-in-pytorch/
# used - https://www.geeksforgeeks.org/deep-learning/implementing-an-autoencoder-in-pytorch/ 


# Refrance List


1.
2.
3.
4.
5.
6.
7.
